# Lección 3: Elementos básicos de Spark — RDD, Transformaciones y Acciones
### Módulo 9 · Retail Analytics Pipeline · RetailMax E-commerce

---

## 📌 Recap de Lección 2

En la **Lección 2** realizamos la configuración inicial del entorno Spark:
- Instalamos PySpark en nuestro entorno virtual `.venv`
- Creamos una `SparkSession` con configuración local (`local[*]`)
- Cargamos los datos de Fashion-MNIST en memoria
- Creamos nuestro **primer RDD básico** con `sc.parallelize()`

## 🎯 ¿Qué haremos en esta Lección 3?

En esta lección profundizaremos en las **operaciones fundamentales** sobre RDDs:

| Sección | Tema | Operaciones |
|---------|------|-------------|
| 0 | Configuración inicial | `SparkSession`, carga de datos, `parallelize` |
| 1 | Transformación `map` | Aplicar funciones elemento a elemento |
| 2 | Transformación `filter` | Seleccionar subconjuntos por condición |
| 3 | Transformación `flatMap` | Expandir y aplanar colecciones |
| 4 | `distinct` y `sortBy` | Eliminar duplicados y ordenar |
| 5 | Pair RDDs y `reduceByKey` | Agrupación y agregación por clave |
| 6 | Acciones: `sum`, `mean`, `stdev` | Disparar la ejecución del DAG |
| 7 | Linaje del RDD (DAG) | Entender la evaluación perezosa |
| 8 | `cache()` y `persist()` | Reutilizar RDDs sin recalcular |
| 9 | Cierre y resumen | `spark.stop()`, tabla de referencia |

## 🔭 Conexión con Lección 4

Al finalizar esta lección habremos dominado los RDDs como capa de bajo nivel. En la **Lección 4** daremos el salto a **DataFrames y Spark SQL**, la API de alto nivel que añade un esquema estructurado y habilita optimizaciones automáticas a través del **Catalyst Optimizer**.

> 💡 **Dataset**: Fashion-MNIST — 70,000 imágenes 28×28 px de 10 categorías de ropa. Empresa ficticia: **RetailMax** (e-commerce).

---
## Sección 0: Configuración inicial

Antes de trabajar con RDDs necesitamos:
1. Importar las librerías necesarias
2. Crear la `SparkSession` (igual que en Lección 2)
3. Cargar Fashion-MNIST en memoria
4. Construir el RDD base

In [1]:
# ============================================================
# SECCIÓN 0: CONFIGURACIÓN INICIAL
# ============================================================

# --- 0.1 Importaciones ---
import os                          # Operaciones del sistema operativo
import time                        # Medición de tiempos de ejecución
import numpy as np                 # Operaciones numéricas vectorizadas
import pandas as pd                # Tablas y análisis exploratorio
import matplotlib.pyplot as plt    # Visualizaciones
import matplotlib.patches as mpatches  # Leyendas en gráficos

# PySpark
from pyspark.sql import SparkSession
from pyspark import StorageLevel    # Niveles de persist() para Sección 8

print("✅ Librerías importadas correctamente")

# --- 0.2 Crear SparkSession (igual que Lección 2) ---
spark = (
    SparkSession.builder
    .appName("RetailMax_Leccion3_RDD")       # Nombre de la aplicación
    .master("local[*]")                       # Usa todos los núcleos disponibles
    .config("spark.driver.memory", "2g")      # 2 GB para el driver
    .config("spark.executor.memory", "2g")    # 2 GB para los executors
    .config("spark.sql.shuffle.partitions", "4")  # Reducir particiones para datos locales
    .config("spark.ui.showConsoleProgress", "false")  # Limpiar la salida en consola
    .getOrCreate()                            # Crea o reutiliza sesión existente
)

# Acceder al SparkContext (nivel bajo, maneja RDDs)
sc = spark.sparkContext
sc.setLogLevel("WARN")  # Reducir verbosidad de logs

print(f"✅ SparkSession creada: {spark.version}")
print(f"   App Name : {sc.appName}")
print(f"   Master   : {sc.master}")

✅ Librerías importadas correctamente
✅ SparkSession creada: 4.1.1
   App Name : RetailMax_Leccion3_RDD
   Master   : local[*]


In [2]:
# ============================================================
# SECCIÓN 0.3: Carga de Fashion-MNIST
# Se intenta en orden: TensorFlow → PyTorch → scikit-learn → datos sintéticos
# ============================================================

# Mapa de etiquetas de Fashion-MNIST
LABEL_NAMES = {
    0: "T-shirt/top",
    1: "Trouser",
    2: "Pullover",
    3: "Dress",
    4: "Coat",
    5: "Sandal",
    6: "Shirt",
    7: "Sneaker",
    8: "Bag",
    9: "Ankle boot"
}

train_images = train_labels = test_images = test_labels = None
fuente = None

# --- Intento 1: TensorFlow/Keras ---
try:
    import tensorflow as tf
    (train_images, train_labels), (test_images, test_labels) = \
        tf.keras.datasets.fashion_mnist.load_data()
    fuente = "TensorFlow/Keras"
    print(f"✅ Fashion-MNIST cargado desde {fuente}")
except Exception as e:
    print(f"⚠️  TensorFlow no disponible: {e}")

# --- Intento 2: PyTorch/torchvision ---
if train_images is None:
    try:
        import torchvision
        import torchvision.transforms as transforms
        transform = transforms.ToTensor()
        _train = torchvision.datasets.FashionMNIST(
            root="./data", train=True, download=True, transform=transform)
        _test  = torchvision.datasets.FashionMNIST(
            root="./data", train=False, download=True, transform=transform)
        train_images = np.array([img.numpy().squeeze() for img, _ in _train])
        train_labels = np.array([lbl for _, lbl in _train])
        test_images  = np.array([img.numpy().squeeze() for img, _ in _test])
        test_labels  = np.array([lbl for _, lbl in _test])
        fuente = "PyTorch/torchvision"
        print(f"✅ Fashion-MNIST cargado desde {fuente}")
    except Exception as e:
        print(f"⚠️  PyTorch no disponible: {e}")

# --- Intento 3: scikit-learn (fetch_openml) ---
if train_images is None:
    try:
        from sklearn.datasets import fetch_openml
        print("⏳ Descargando Fashion-MNIST desde OpenML (puede tardar ~60s)...")
        fmnist = fetch_openml("Fashion-MNIST", version=1, as_frame=False, parser="liac-arff")
        X = fmnist.data.astype(np.float32)
        y = fmnist.target.astype(np.int32)
        train_images = X[:60000].reshape(-1, 28, 28)
        train_labels = y[:60000]
        test_images  = X[60000:].reshape(-1, 28, 28)
        test_labels  = y[60000:]
        fuente = "scikit-learn/OpenML"
        print(f"✅ Fashion-MNIST cargado desde {fuente}")
    except Exception as e:
        print(f"⚠️  scikit-learn OpenML no disponible: {e}")

# --- Intento 4: Datos sintéticos (fallback garantizado) ---
if train_images is None:
    print("🔄 Generando datos sintéticos (Fashion-MNIST simulado)...")
    np.random.seed(42)
    # 60,000 imágenes de entrenamiento sintéticas
    train_images = np.random.randint(0, 256, (60000, 28, 28), dtype=np.uint8)
    train_labels = np.random.randint(0, 10,  60000,            dtype=np.int32)
    # 10,000 imágenes de prueba sintéticas
    test_images  = np.random.randint(0, 256, (10000, 28, 28), dtype=np.uint8)
    test_labels  = np.random.randint(0, 10,  10000,           dtype=np.int32)
    fuente = "Datos sintéticos (fallback)"
    print(f"✅ Datos sintéticos generados desde {fuente}")

print(f"\n📊 Forma del conjunto de entrenamiento : {train_images.shape}")
print(f"📊 Forma del conjunto de prueba        : {test_images.shape}")
print(f"📊 Fuente de datos                     : {fuente}")

✅ Fashion-MNIST cargado desde TensorFlow/Keras

📊 Forma del conjunto de entrenamiento : (60000, 28, 28)
📊 Forma del conjunto de prueba        : (10000, 28, 28)
📊 Fuente de datos                     : TensorFlow/Keras


In [3]:
# ============================================================
# SECCIÓN 0.4: Preparar la lista de diccionarios
# Usamos 10,000 imágenes de train + 2,000 de test = 12,000 total
# ============================================================

N_TRAIN = 10_000  # Primeras 10,000 imágenes de entrenamiento
N_TEST  =  2_000  # Primeros 2,000 imágenes de prueba

data = []  # Lista de diccionarios que irá al RDD

# --- Procesar imágenes de entrenamiento ---
for i in range(N_TRAIN):
    img   = train_images[i]              # Imagen 28x28 (uint8)
    label = int(train_labels[i])         # Etiqueta numérica 0-9
    # Aplanar y normalizar al rango [0, 1]
    pixels = (img.flatten().astype(np.float32) / 255.0).tolist()
    data.append({
        "image_id"   : i,                    # Identificador único
        "label"      : label,                # Etiqueta numérica
        "label_name" : LABEL_NAMES[label],   # Nombre de la categoría
        "pixels"     : pixels,               # Lista de 784 floats [0,1]
        "split"      : "train"               # Partición del dataset
    })

# --- Procesar imágenes de prueba ---
for j in range(N_TEST):
    img   = test_images[j]
    label = int(test_labels[j])
    pixels = (img.flatten().astype(np.float32) / 255.0).tolist()
    data.append({
        "image_id"   : N_TRAIN + j,          # IDs continúan desde 10,000
        "label"      : label,
        "label_name" : LABEL_NAMES[label],
        "pixels"     : pixels,
        "split"      : "test"
    })

TOTAL = len(data)
print(f"✅ Lista de diccionarios preparada")
print(f"   Total de registros : {TOTAL:,}")
print(f"   Train              : {N_TRAIN:,}")
print(f"   Test               : {N_TEST:,}")
print(f"\n🔍 Ejemplo de un registro (sin los píxeles):")
ejemplo = {k: v for k, v in data[0].items() if k != "pixels"}
ejemplo["pixels"] = f"[{data[0]['pixels'][0]:.4f}, {data[0]['pixels'][1]:.4f}, ... ] (784 valores)"
for k, v in ejemplo.items():
    print(f"   {k:12s}: {v}")

✅ Lista de diccionarios preparada
   Total de registros : 12,000
   Train              : 10,000
   Test               : 2,000

🔍 Ejemplo de un registro (sin los píxeles):
   image_id    : 0
   label       : 9
   label_name  : Ankle boot
   split       : train
   pixels      : [0.0000, 0.0000, ... ] (784 valores)


In [4]:
# ============================================================
# SECCIÓN 0.5: Crear el RDD base con sc.parallelize()
# ============================================================

NUM_SLICES = 4  # Número de particiones (una por núcleo en equipo estándar)

# sc.parallelize() distribuye la colección Python en NUM_SLICES particiones
rdd_base = sc.parallelize(data, numSlices=NUM_SLICES)

# Verificar el RDD creado
num_registros   = rdd_base.count()          # Acción: cuenta todos los elementos
num_particiones = rdd_base.getNumPartitions()  # Número de particiones reales

print(f"✅ RDD creado con {num_registros:,} registros en {num_particiones} particiones")
print(f"\n🔍 Primer registro del RDD (sin píxeles):")

primer = rdd_base.first()  # Acción: trae el primer elemento al driver
for k, v in primer.items():
    if k == "pixels":
        print(f"   {k:12s}: [{v[0]:.4f}, {v[1]:.4f}, ..., {v[-1]:.4f}] ({len(v)} valores)")
    else:
        print(f"   {k:12s}: {v}")

Py4JJavaError: An error occurred while calling z:org.apache.spark.api.python.PythonRDD.collectAndServe.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 2 in stage 0.0 failed 1 times, most recent failure: Lost task 2.0 in stage 0.0 (TID 2) (Urzua executor driver): org.apache.spark.SparkException: Python worker exited unexpectedly (crashed). Consider setting 'spark.sql.execution.pyspark.udf.faulthandler.enabled' or'spark.python.worker.faulthandler.enabled' configuration to 'true' for the better Python traceback.
	at org.apache.spark.api.python.BasePythonRunner$ReaderIterator$$anonfun$1.applyOrElse(PythonRunner.scala:678)
	at org.apache.spark.api.python.BasePythonRunner$ReaderIterator$$anonfun$1.applyOrElse(PythonRunner.scala:663)
	at scala.runtime.AbstractPartialFunction.apply(AbstractPartialFunction.scala:35)
	at org.apache.spark.api.python.PythonRunner$$anon$3.read(PythonRunner.scala:1034)
	at org.apache.spark.api.python.PythonRunner$$anon$3.read(PythonRu

---
## Sección 1: Transformaciones — `map`

### ¿Qué es `map`?

`map` aplica una función a **cada elemento** del RDD y devuelve un nuevo RDD con los resultados transformados. La correspondencia es **1 entrada → 1 salida**.

**Analogía**: imagina una **línea de ensamblaje** (conveyor belt). Cada pieza entra por un extremo, pasa por una estación de transformación, y sale transformada por el otro extremo. Ninguna pieza se crea ni se elimina, solo se transforma.

```python
# Sintaxis general
rdd_nuevo = rdd.map(lambda x: función(x))
```

### ⚡ ¡Es LAZY! (Evaluación perezosa)

Cuando escribes `rdd.map(...)`, Spark **NO** ejecuta nada todavía. Solo registra la transformación en el plan de ejecución (DAG). La ejecución real ocurre cuando se llama una **acción** como `.count()`, `.collect()`, `.take(n)`, etc.

### Mini-ejemplo numérico:

```
RDD original  : [1, 2, 3, 4, 5]
Transformación: map(lambda x: x * 2)
RDD resultado : [2, 4, 6, 8, 10]
                ↑  ↑  ↑  ↑  ↑
                Cada elemento se duplica independientemente
```

Cada worker procesa su partición de forma **paralela**:
```
Partición 0: [1, 2] → [2, 4]
Partición 1: [3, 4] → [6, 8]
Partición 2: [5]    → [10]
```

In [ ]:
# ============================================================
# SECCIÓN 1: TRANSFORMACIÓN map
# ============================================================

print("=" * 60)
print("PASO 1: Extraer metadata con map")
print("=" * 60)

# map: extraer solo los campos de metadata (sin los 784 píxeles)
# Esto es LAZY — solo registra la transformación, no ejecuta aún
rdd_metadata = rdd_base.map(lambda r: (
    r["image_id"],    # ID único de la imagen
    r["label"],       # Etiqueta numérica (0-9)
    r["label_name"],  # Nombre de la categoría
    r["split"]        # "train" o "test"
))

# .take(2) es una acción → DISPARA la ejecución del DAG
print("\n🔍 Primeros 2 registros de rdd_metadata:")
for registro in rdd_metadata.take(2):
    print(f"   image_id={registro[0]:5d} | label={registro[1]} | "
          f"label_name={registro[2]:12s} | split={registro[3]}")

print(f"\n📝 Cambio: de diccionarios con 788 claves → tuplas de 4 campos")
print(f"   Tamaño RDD base    : {rdd_base.count():,} registros (diccionarios con pixels)")
print(f"   Tamaño rdd_metadata: {rdd_metadata.count():,} registros (tuplas ligeras)")

print()
print("=" * 60)
print("PASO 2: Calcular la media de píxeles por imagen con map")
print("=" * 60)

# map: calcular la media de los 784 píxeles de cada imagen
# Resultado: RDD de tuplas (image_id, pixel_mean)
rdd_pixel_means = rdd_base.map(lambda r: (
    r["image_id"],
    sum(r["pixels"]) / len(r["pixels"])  # Media aritmética manual (compatible con Spark)
))

print("\n🔍 Primeros 2 registros de rdd_pixel_means:")
for image_id, mean_val in rdd_pixel_means.take(2):
    print(f"   image_id={image_id:5d} | pixel_mean={mean_val:.6f}")
print("\n📝 Cambio: cada imagen (784 píxeles) → un solo número (la media)")

print()
print("=" * 60)
print("PASO 3: Renormalizar píxeles de [0,1] a [-1,1] con map")
print("=" * 60)

# map: renormalizar píxeles aplicando la fórmula pixel_norm = pixel * 2 - 1
#   Si pixel = 0.0  → norm = -1.0  (negro)
#   Si pixel = 0.5  → norm =  0.0  (gris)
#   Si pixel = 1.0  → norm =  1.0  (blanco)
rdd_normalized = rdd_base.map(lambda r: {
    **r,  # Copiar todos los campos existentes del diccionario
    "pixels": [p * 2 - 1 for p in r["pixels"]]  # Renormalizar cada píxel
})

print("\n🔍 Verificación de la renormalización (primeros 5 píxeles):")
original   = rdd_base.first()["pixels"][:5]
normalizado = rdd_normalized.first()["pixels"][:5]
print("   Rango original  [0,1] :", [f"{p:.4f}" for p in original])
print("   Rango normaliz. [-1,1]:", [f"{p:.4f}" for p in normalizado])
print("\n📝 Fórmula: pixel_norm = pixel * 2 - 1")
print("   Útil para redes neuronales que esperan entradas centradas en 0")

---
## Sección 2: Transformaciones — `filter`

### ¿Qué es `filter`?

`filter` selecciona únicamente los elementos del RDD que cumplen una **condición booleana**. Los elementos que no cumplen la condición son descartados. La correspondencia es **1 entrada → 0 o 1 salida**.

**Analogía**: imagina un **colador** de cocina. Cuando viertes agua con pasta, solo la pasta (elementos que pasan la condición) queda en el colador. El agua (elementos descartados) se va por los agujeros.

```python
# Sintaxis general
rdd_filtrado = rdd.filter(lambda x: condición_booleana)
```

### Mini-ejemplo numérico:

```
RDD original  : [1, 2, 3, 4, 5]
Condición     : filter(lambda x: x > 3)
RDD resultado : [4, 5]
                   ↑  ↑
                   Solo los que cumplen x > 3

Evaluación por elemento:
  1 > 3? ❌ descartado
  2 > 3? ❌ descartado
  3 > 3? ❌ descartado
  4 > 3? ✅ conservado
  5 > 3? ✅ conservado
```

In [ ]:
# ============================================================
# SECCIÓN 2: TRANSFORMACIÓN filter
# ============================================================

total = rdd_base.count()  # Total de registros en el RDD base

print("=" * 60)
print("PASO 1: Filtrar solo imágenes de la clase 'Bag' (label == 8)")
print("=" * 60)

# filter: conservar solo las imágenes cuya etiqueta sea 8 (Bag)
rdd_bags = rdd_base.filter(lambda r: r["label"] == 8)

total_bags = rdd_bags.count()
pct_bags   = total_bags / total * 100

print(f"   Registros ANTES del filtro  : {total:,}")
print(f"   Registros DESPUÉS del filtro: {total_bags:,}")
print(f"   Porcentaje que representa   : {pct_bags:.2f}%")
print(f"   (Esperado ≈ 10% en dataset balanceado)")

print("\n🔍 Primer registro de rdd_bags:")
first_bag = rdd_bags.first()
print(f"   image_id={first_bag['image_id']} | label={first_bag['label']} | "
      f"label_name={first_bag['label_name']}")

print()
print("=" * 60)
print("PASO 2: Filtrar imágenes 'brillantes' (pixel_mean > 0.5)")
print("=" * 60)

# Nota: rdd_pixel_means tiene tuplas (image_id, pixel_mean)
# Filtramos aquellas con media de píxel mayor a 0.5
# pixel_mean > 0.5 indica imágenes más claras (fondo blanco o colores claros)
rdd_bright = rdd_pixel_means.filter(lambda x: x[1] > 0.5)

total_pm     = rdd_pixel_means.count()
total_bright = rdd_bright.count()
pct_bright   = total_bright / total_pm * 100

print(f"   Registros ANTES del filtro  : {total_pm:,}")
print(f"   Registros DESPUÉS del filtro: {total_bright:,}")
print(f"   Porcentaje que representa   : {pct_bright:.2f}%")
print("\n🔍 Primeros 3 registros brillantes:")
for image_id, mean_val in rdd_bright.take(3):
    print(f"   image_id={image_id:5d} | pixel_mean={mean_val:.6f}")

print()
print("=" * 60)
print("PASO 3: Filtrar solo registros del split 'test'")
print("=" * 60)

# filter: conservar solo las imágenes del conjunto de prueba
rdd_test = rdd_base.filter(lambda r: r["split"] == "test")

total_test = rdd_test.count()
pct_test   = total_test / total * 100

print(f"   Registros ANTES del filtro  : {total:,}")
print(f"   Registros DESPUÉS del filtro: {total_test:,}")
print(f"   Porcentaje que representa   : {pct_test:.2f}%")
print(f"   (Esperado = {N_TEST:,} registros de test)")

print()
print("📊 RESUMEN DE FILTROS:")
resumen_filtros = pd.DataFrame([
    {"Filtro": "label == 8 (Bag)",   "Registros": total_bags,   "% del total": pct_bags},
    {"Filtro": "pixel_mean > 0.5",   "Registros": total_bright, "% del total": pct_bright},
    {"Filtro": "split == 'test'",    "Registros": total_test,   "% del total": pct_test},
])
resumen_filtros["% del total"] = resumen_filtros["% del total"].map("{:.2f}%".format)
print(resumen_filtros.to_string(index=False))

---
## Sección 3: Transformaciones — `flatMap`

### ¿Qué es `flatMap`?

`flatMap` aplica una función a cada elemento del RDD, pero a diferencia de `map`, la función puede devolver **cero o más elementos** por cada entrada. Los resultados de todas las entradas se **aplanan** en un único RDD de salida.

| Transformación | Entradas → Salidas | Relación |
|---|---|---|
| `map` | `N` entradas → `N` salidas | 1:1 |
| `flatMap` | `N` entradas → `0, 1, o más` salidas | 1:N |

**Analogía**: 
- `map` es **doblar una hoja** de papel: tienes la misma cantidad de hojas, solo cambia su forma
- `flatMap` es **desdoblar** sobres de cartas: abres cada sobre, sacas las cartas que contiene, y las apilas todas juntas en una sola pila plana

### Mini-ejemplo numérico:

```python
# Con map: cada string se convierte en UNA lista
rdd = sc.parallelize(["hola mundo", "big data"])
rdd.map(lambda s: s.split())    → [["hola", "mundo"], ["big", "data"]]  # 2 listas

# Con flatMap: los resultados se APLANAN en un único nivel
rdd.flatMap(lambda s: s.split()) → ["hola", "mundo", "big", "data"]    # 4 strings
```

```
Antes (flatMap):
  Entrada 1: "hola mundo"  → ["hola", "mundo"]
  Entrada 2: "big data"    → ["big", "data"]
                              ↓  Aplanar
Después (flatMap):            ["hola", "mundo", "big", "data"]
```

In [ ]:
# ============================================================
# SECCIÓN 3: TRANSFORMACIÓN flatMap
# ============================================================

print("=" * 60)
print("PASO 1: Expandir cada imagen en tuplas (image_id, pixel_idx, pixel_val)")
print("        usando los primeros 10 píxeles de cada imagen")
print("=" * 60)

# flatMap: cada imagen genera una LISTA de 10 tuplas (una por píxel)
# La función lambda devuelve una lista → flatMap la aplana
rdd_pixels_flat = rdd_base.flatMap(lambda r: [
    (r["image_id"], pixel_idx, r["pixels"][pixel_idx])
    for pixel_idx in range(10)  # Solo los primeros 10 píxeles por imagen
])

print()
print("PASO 2: Contar el total de tuplas generadas")
count_original = rdd_base.count()       # Número de imágenes originales
count_flat     = rdd_pixels_flat.count()  # Número de tuplas tras flatMap

print(f"   RDD original (imágenes) : {count_original:,} registros")
print(f"   RDD flat (tuplas pixel) : {count_flat:,} registros")
print(f"   Factor de expansión     : {count_flat // count_original}x (10 píxeles/imagen)")
print(f"   Verificación            : {count_original:,} × 10 = {count_original * 10:,} ✅" 
      if count_flat == count_original * 10 else "   ⚠️ No coincide")

print()
print("PASO 3: Primeros 5 elementos del RDD aplanado")
print("Formato: (image_id, pixel_index, pixel_value)")
print("-" * 50)
for image_id, pixel_idx, pixel_val in rdd_pixels_flat.take(5):
    print(f"   ({image_id:5d}, pixel[{pixel_idx:2d}], {pixel_val:.6f})")

print()
print("📝 DIFERENCIA map vs flatMap en este ejemplo:")
print("   map(lambda r: [(id, i, p) ...])  → RDD de LISTAS: [[t1,t2,...], [t1,t2,...], ...]")
print("   flatMap(lambda r: [(id, i, p)...]) → RDD PLANO : [t1, t2, ..., t1, t2, ...]")
print(f"   map daría {count_original:,} listas de 10 elementos cada una")
print(f"   flatMap da {count_flat:,} tuplas individuales")

---
## Sección 4: Transformaciones — `distinct` y `sortBy`

### `distinct` — Eliminar duplicados

`distinct()` devuelve un nuevo RDD con los **elementos únicos** del RDD original. Internamente hace un shuffle entre particiones para encontrar y eliminar duplicados a escala.

```python
rdd = sc.parallelize([1, 2, 2, 3, 3, 3, 4])
rdd.distinct().collect()  # → [1, 2, 3, 4]
```

**Cuándo usarlo**: extraer categorías únicas, identificar IDs duplicados, limpiar datos.

---

### `sortBy` — Ordenar por una clave

`sortBy(keyfunc, ascending=True)` ordena los elementos del RDD según la clave que devuelve `keyfunc`.

```python
rdd = sc.parallelize([("b", 2), ("a", 3), ("c", 1)])
rdd.sortBy(lambda x: x[1]).collect()  # → [("c",1), ("b",2), ("a",3)]  ← por valor
rdd.sortBy(lambda x: x[0]).collect()  # → [("a",3), ("b",2), ("c",1)]  ← por clave
```

**Cuándo usarlo**: rankings, reportes ordenados, visualizaciones de barras.

In [ ]:
# ============================================================
# SECCIÓN 4: distinct y sortBy
# ============================================================

print("=" * 60)
print("PASO 1: Extraer categorías únicas con distinct()")
print("=" * 60)

# map para extraer solo el label_name → luego distinct() para valores únicos
rdd_categorias = (
    rdd_base
    .map(lambda r: r["label_name"])  # Extraer nombre de categoría
    .distinct()                       # Eliminar duplicados
)

categorias_unicas = sorted(rdd_categorias.collect())  # collect() trae al driver
print(f"   Número de categorías únicas: {len(categorias_unicas)}")
print(f"   Categorías: {categorias_unicas}")
print(f"   ✅ Correcto: Fashion-MNIST tiene exactamente 10 categorías")

print()
print("=" * 60)
print("PASO 2: Contar imágenes por clase con map + reduceByKey")
print("=" * 60)

# map: cada registro genera una tupla (nombre_clase, 1)
# reduceByKey: suma los unos agrupando por clave (nombre_clase)
rdd_label_counts = (
    rdd_base
    .map(lambda r: (r["label_name"], 1))  # (clase, 1) por cada imagen
    .reduceByKey(lambda a, b: a + b)       # Sumar conteos por clase
)

print("   Conteo por clase (sin ordenar):")
for clase, conteo in sorted(rdd_label_counts.collect(), key=lambda x: x[0]):
    print(f"   {clase:15s}: {conteo:,}")

print()
print("=" * 60)
print("PASO 3: Ordenar de mayor a menor frecuencia con sortBy")
print("=" * 60)

# sortBy(lambda x: x[1], ascending=False): ordena por el conteo, descendente
rdd_sorted = rdd_label_counts.sortBy(lambda x: x[1], ascending=False)

ranking = rdd_sorted.collect()  # Trae al driver los resultados ordenados

print("   Ranking de clases por frecuencia (mayor → menor):")
print(f"   {'Pos':>4} {'Clase':15s} {'Imágenes':>10} {'%':>8}")
print("   " + "-" * 42)
total_imagenes = sum(c for _, c in ranking)
for pos, (clase, conteo) in enumerate(ranking, 1):
    barra = "█" * int(conteo / total_imagenes * 30)
    print(f"   {pos:>4} {clase:15s} {conteo:>10,} {conteo/total_imagenes*100:>7.1f}% {barra}")

In [ ]:
# ============================================================
# SECCIÓN 4 (cont.): Visualización — Bar chart de distribución de clases
# ============================================================

# Preparar datos para el gráfico
clases  = [clase  for clase, _ in ranking]
conteos = [conteo for _, conteo in ranking]

# Paleta de colores para las 10 categorías
colores = plt.cm.tab10(np.linspace(0, 1, len(clases)))

fig, ax = plt.subplots(figsize=(10, 6))

# Gráfico de barras horizontales (facilita lectura de etiquetas largas)
barras = ax.barh(clases[::-1], conteos[::-1], color=colores, edgecolor="white", linewidth=0.5)

# Etiquetas de valor al final de cada barra
for barra, conteo in zip(barras, conteos[::-1]):
    ax.text(
        barra.get_width() + 10,           # Posición X (ligeramente a la derecha)
        barra.get_y() + barra.get_height() / 2,  # Centrado en Y
        f"{conteo:,}",                    # Texto con separador de miles
        va="center", ha="left", fontsize=9
    )

# Línea de referencia: distribución perfectamente balanceada
total_total = sum(conteos)
media_ideal = total_total / len(clases)
ax.axvline(media_ideal, color="red", linestyle="--", linewidth=1.2,
           label=f"Media ideal ({media_ideal:.0f} imgs/clase)")

# Estética
ax.set_xlabel("Número de imágenes", fontsize=11)
ax.set_title(
    f"Distribución de clases en Fashion-MNIST\n"
    f"(RetailMax · {total_total:,} imágenes · Lección 3)",
    fontsize=13, fontweight="bold"
)
ax.legend(fontsize=9)
ax.set_xlim(0, max(conteos) * 1.15)  # Espacio para las etiquetas
ax.grid(axis="x", linestyle=":", alpha=0.5)
ax.spines[["top", "right"]].set_visible(False)

plt.tight_layout()
plt.savefig("distribucion_clases.png", dpi=120, bbox_inches="tight")
plt.show()
print("\n📊 Gráfico guardado como 'distribucion_clases.png'")
print("\n💡 Observación: si el dataset está balanceado, todas las barras deberían")
print("   tener la misma longitud (≈ la línea roja punteada).")

---
## Sección 5: Pair RDDs y `reduceByKey`

### ¿Qué es un Pair RDD?

Un **Pair RDD** es un RDD cuyos elementos son **tuplas de dos elementos** `(clave, valor)`. Esta estructura habilita operaciones especiales de agrupación y agregación.

```python
# RDD regular:   ["hola", "mundo", "spark"]
# Pair RDD:      [("a", 1), ("b", 2), ("a", 3)]
```

### Operaciones especiales en Pair RDDs:

| Operación | Descripción | Eficiencia |
|---|---|---|
| `reduceByKey(f)` | Agrega valores con la misma clave usando `f` | ✅ Alta (combina localmente primero) |
| `groupByKey()` | Agrupa todos los valores de cada clave en una lista | ⚠️ Baja (mueve todos los datos) |
| `mapValues(f)` | Aplica `f` solo a los valores, sin tocar las claves | ✅ Alta (no hace shuffle) |
| `sortByKey()` | Ordena por clave | Media |
| `join(other_rdd)` | Une dos Pair RDDs por clave común | Depende |

### ⚠️ `reduceByKey` vs `groupByKey`:

```
groupByKey():  Envía TODOS los valores al shuffle → memoria alta
               ("a", [v1, v2, v3, v4, ...todos...]) → suma al final

reduceByKey(): Combina localmente en cada partición PRIMERO → luego shuffle
               Partición 1: ("a", v1+v2) → shuffle → ("a", v1+v2+v3+v4)
               Partición 2: ("a", v3+v4) ↗
```

**Regla de oro**: Prefiere siempre `reduceByKey` sobre `groupByKey` cuando solo necesites una agregación (suma, máximo, mínimo, etc.).

In [ ]:
# ============================================================
# SECCIÓN 5: PAIR RDDs y reduceByKey
# ============================================================

print("=" * 60)
print("PASO 1: Crear Pair RDD (label_name, pixel_mean)")
print("=" * 60)

# map: cada imagen → tupla (nombre_clase, media_pixel)
# Este es un Pair RDD: clave = nombre_clase, valor = pixel_mean
rdd_pair_means = rdd_base.map(lambda r: (
    r["label_name"],                               # Clave: nombre de la clase
    sum(r["pixels"]) / len(r["pixels"])            # Valor: media de los 784 píxeles
))

print("🔍 Primeros 4 elementos del Pair RDD:")
for clase, mean_val in rdd_pair_means.take(4):
    print(f"   ({clase!r:15s}, {mean_val:.6f})")

print()
print("=" * 60)
print("PASO 2: Calcular (suma, conteo) por clase con reduceByKey")
print("=" * 60)

# Estrategia: crear Pair RDD con (clase, (pixel_mean, 1))
# Luego reduceByKey suma ambos componentes de la tupla
# Finalmente mapValues divide suma / conteo para obtener la media real

# Paso 2a: Crear (clase, (suma_pixel, conteo))
rdd_pair_sum_count = rdd_base.map(lambda r: (
    r["label_name"],                                      # Clave
    (sum(r["pixels"]) / len(r["pixels"]), 1)              # Valor: (media, 1)
))

# Paso 2b: reduceByKey suma ambos componentes del par
# (suma1, cnt1) + (suma2, cnt2) → (suma1+suma2, cnt1+cnt2)
rdd_aggregated = rdd_pair_sum_count.reduceByKey(
    lambda a, b: (a[0] + b[0], a[1] + b[1])   # Sumar suma y conteo por separado
)

print("   Valores intermedios (suma_media, conteo) por clase:")
for clase, (suma, cnt) in sorted(rdd_aggregated.collect(), key=lambda x: x[0]):
    print(f"   {clase:15s}: suma={suma:10.4f}, conteo={cnt:,}")

print()
print("=" * 60)
print("PASO 3: Calcular media real con mapValues (suma / conteo)")
print("=" * 60)

# mapValues: aplica la función solo al valor, sin tocar la clave
# v = (suma_medias, conteo) → media_real = suma_medias / conteo
rdd_class_means = (
    rdd_aggregated
    .mapValues(lambda v: v[0] / v[1])   # Media = suma / conteo
    .sortBy(lambda x: x[1], ascending=False)  # Ordenar de mayor a menor media
)

print()
print("=" * 60)
print("PASO 4: Tabla: clase → pixel_mean_promedio")
print("=" * 60)

resultados_clase = rdd_class_means.collect()

print(f"\n   {'Clase':15s} {'Media Píxeles':>14} {'Brillo relativo':>16}")
print("   " + "-" * 50)
max_mean = max(m for _, m in resultados_clase)
for clase, media in resultados_clase:
    barra = "█" * int(media / max_mean * 20)
    print(f"   {clase:15s} {media:>14.6f} {barra}")

clase_mas_brillante = resultados_clase[0]
clase_mas_oscura    = resultados_clase[-1]

print(f"\n💡 Interpretación para RetailMax:")
print(f"   Clase MÁS brillante  : '{clase_mas_brillante[0]}' (media={clase_mas_brillante[1]:.4f})")
print(f"   → Esta categoría tiende a tener fondos o colores más claros en las imágenes")
print(f"   Clase MÁS oscura     : '{clase_mas_oscura[0]}' (media={clase_mas_oscura[1]:.4f})")
print(f"   → Esta categoría tiene más píxeles oscuros (prendas o fondos oscuros)")

---
## Sección 6: Acciones — `collect`, `sum`, `mean`, `stdev`

### ¿Qué son las Acciones?

Las **acciones** son operaciones que **disparan la ejecución** del DAG completo y devuelven un resultado al driver (o escriben a almacenamiento externo). Hasta que no se llama una acción, Spark solo acumula transformaciones en el plan de ejecución.

| Acción | Qué devuelve | Cuándo usarla |
|---|---|---|
| `collect()` | Lista Python con todos los elementos | ⚠️ Solo en datasets pequeños |
| `take(n)` | Lista Python con los primeros `n` elementos | Ver muestras |
| `count()` | Número entero con el conteo total | Saber el tamaño |
| `first()` | El primer elemento | Inspección rápida |
| `sum()` | Suma de todos los elementos numéricos | Totales |
| `mean()` | Media aritmética | Estadísticas descriptivas |
| `stdev()` | Desviación estándar | Dispersión de datos |
| `max()` / `min()` | Máximo / Mínimo | Valores extremos |
| `saveAsTextFile(path)` | None (escribe a disco/HDFS) | Persistir resultados |

### ⚠️ ADVERTENCIA sobre `collect()`

`collect()` transfiere **TODOS** los datos del cluster al nodo driver. Si tu RDD tiene millones de registros, esto puede:
- Causar `OutOfMemoryError` en el driver
- Saturar la red del cluster
- Hacer inútil el procesamiento distribuido

**Regla**: Usa `collect()` solo cuando estés seguro de que el resultado cabe en memoria del driver.

### Desviación estándar — mini-explicación:

La desviación estándar (σ) mide cuánto se **alejan en promedio** los valores de la media:

$$\sigma = \sqrt{\frac{\sum_{i=1}^{N}(x_i - \mu)^2}{N}}$$

- σ **pequeña** → los valores están muy concentrados cerca de la media
- σ **grande** → los valores están muy dispersos

In [ ]:
# ============================================================
# SECCIÓN 6: ACCIONES — collect, sum, mean, stdev
# ============================================================

print("=" * 60)
print("PASO 1: Extraer solo los valores numéricos de medias de píxeles")
print("=" * 60)

# rdd_pixel_means tiene tuplas (image_id, pixel_mean)
# Extraemos solo el valor numérico (float) de la media
rdd_values = rdd_pixel_means.map(lambda x: float(x[1]))
print(f"   RDD de valores creado (lazy — no ejecutado aún)")
print(f"   Primeros 5 valores: {rdd_values.take(5)}")

print()
print("=" * 60)
print("PASO 2: count() — número total de elementos")
print("=" * 60)

t0 = time.time()          # Registrar tiempo de inicio
n  = rdd_values.count()   # ACCIÓN: dispara ejecución del DAG
t1 = time.time()          # Registrar tiempo de fin

print(f"   count()         = {n:,} elementos")
print(f"   Tiempo de cómputo: {t1 - t0:.3f} segundos")

print()
print("=" * 60)
print("PASO 3: sum() — suma total de todas las medias")
print("=" * 60)

total_sum = rdd_values.sum()   # ACCIÓN: suma todos los flotantes
print(f"   sum()           = {total_sum:.6f}")
print(f"   Interpretación  : Suma de las medias de píxeles de las {n:,} imágenes")

print()
print("=" * 60)
print("PASO 4: mean() — media global")
print("=" * 60)

global_mean = rdd_values.mean()   # ACCIÓN: calcula la media
print(f"   mean()          = {global_mean:.6f}")
print(f"   Verificación    : sum()/count() = {total_sum/n:.6f}")
print(f"   ✅ Coinciden" if abs(global_mean - total_sum/n) < 1e-9 else "   ⚠️ No coinciden")

print()
print("=" * 60)
print("PASO 5: stdev() — desviación estándar")
print("=" * 60)

stdev = rdd_values.stdev()   # ACCIÓN: calcula desviación estándar poblacional
print(f"   stdev()         = {stdev:.6f}")
print(f"   Fórmula: σ = √(Σ(xi - μ)² / N)")
print(f"   Interpretación  : Las medias de píxeles varían ±{stdev:.4f} respecto a la media global")

print()
print("=" * 60)
print("PASO 6: max() y min()")
print("=" * 60)

val_max = rdd_values.max()   # ACCIÓN: valor máximo
val_min = rdd_values.min()   # ACCIÓN: valor mínimo
print(f"   max()           = {val_max:.6f} (imagen con píxeles más claros en promedio)")
print(f"   min()           = {val_min:.6f} (imagen con píxeles más oscuros en promedio)")
print(f"   Rango           = {val_max - val_min:.6f}")

print()
print("=" * 60)
print("RESUMEN ESTADÍSTICO — Medias de píxeles por imagen")
print("=" * 60)

resumen = pd.DataFrame([{
    "Estadístico" : "count",
    "Valor"       : f"{n:,}",
    "Descripción" : "Número total de imágenes"
}, {
    "Estadístico" : "sum",
    "Valor"       : f"{total_sum:.4f}",
    "Descripción" : "Suma total de medias de píxeles"
}, {
    "Estadístico" : "mean (μ)",
    "Valor"       : f"{global_mean:.6f}",
    "Descripción" : "Media global de las medias de píxeles"
}, {
    "Estadístico" : "stdev (σ)",
    "Valor"       : f"{stdev:.6f}",
    "Descripción" : "Dispersión de las medias de píxeles"
}, {
    "Estadístico" : "min",
    "Valor"       : f"{val_min:.6f}",
    "Descripción" : "Imagen más oscura (media de píxel más baja)"
}, {
    "Estadístico" : "max",
    "Valor"       : f"{val_max:.6f}",
    "Descripción" : "Imagen más clara (media de píxel más alta)"
}])

print(resumen.to_string(index=False))

---
## Sección 7: Linaje del RDD (DAG)

### ¿Qué es el DAG?

El **DAG** (Directed Acyclic Graph — Grafo Dirigido Acíclico) es la representación interna que usa Spark para planificar la ejecución de las transformaciones.

```
Directed  → Las dependencias van en una sola dirección (de padre a hijo)
Acyclic   → No hay ciclos (no puedes volver a un nodo anterior)
Graph     → Los nodos son RDDs y las aristas son transformaciones
```

### ¿Por qué Spark lo usa?

1. **Lazy Evaluation**: Spark acumula transformaciones sin ejecutarlas hasta que se necesita un resultado (acción). Esto permite optimizar el plan completo antes de ejecutar.
2. **Optimización**: Spark puede reordenar operaciones (ej: aplicar filters antes, para reducir datos).
3. **Tolerancia a fallos**: Si un nodo falla, Spark puede reconstruir las particiones perdidas usando el linaje del DAG.

### Transformaciones Narrow vs Wide:

| Tipo | Descripción | Ejemplos | Shuffle? |
|---|---|---|---|
| **Narrow** | Cada partición de salida depende de **una sola** partición de entrada | `map`, `filter`, `flatMap` | ❌ No |
| **Wide** | Cada partición de salida puede depender de **múltiples** particiones | `reduceByKey`, `sortBy`, `distinct` | ✅ Sí |

Las transformaciones **Wide** son más costosas porque requieren **shuffle** (mover datos entre nodos).

### `toDebugString()`

Muestra el linaje completo del RDD en formato de texto, indicando el tipo de dependencia y las particiones.

In [ ]:
# ============================================================
# SECCIÓN 7: LINAJE DEL RDD (DAG)
# ============================================================

print("=" * 60)
print("PASO 1: Construir cadena de transformaciones para el pipeline")
print("=" * 60)

# Construimos un pipeline completo:
# rdd_base → map(metadata) → filter(bags) → map(pixel_mean) → sortBy

# Paso A: Extraer metadata de todas las imágenes
rdd_dag_meta = rdd_base.map(lambda r: {
    "image_id"   : r["image_id"],
    "label"      : r["label"],
    "label_name" : r["label_name"],
    "pixels"     : r["pixels"]
})

# Paso B: Filtrar solo la clase 'Bag' (label == 8)
rdd_dag_bags = rdd_dag_meta.filter(lambda r: r["label"] == 8)

# Paso C: Calcular la media de píxeles de cada imagen de tipo Bag
rdd_dag_means = rdd_dag_bags.map(lambda r: (
    r["image_id"],
    sum(r["pixels"]) / len(r["pixels"])
))

# Paso D: Ordenar por pixel_mean (de más oscuro a más claro)
rdd_dag_sorted = rdd_dag_means.sortBy(lambda x: x[1], ascending=True)

print("   Pipeline construido (aún no ejecutado — lazy evaluation):")
print("   rdd_base → map(metadata) → filter(bags) → map(pixel_mean) → sortBy")

print()
print("=" * 60)
print("PASO 2: Inspeccionar el linaje con toDebugString()")
print("=" * 60)

# toDebugString() devuelve bytes — decodificar a string UTF-8
debug_string = rdd_dag_sorted.toDebugString().decode("utf-8")

print("\n--- toDebugString() ---")
print(debug_string)
print("-" * 40)
print("Nota: La indentación indica el nivel de dependencia (más sangría = más antiguo en el linaje)")

print()
print("=" * 60)
print("PASO 3: DAG representado visualmente como texto ASCII")
print("=" * 60)

print()
print("  DAG del pipeline RetailMax — Análisis de Bags (label=8):")
print()
print("  ┌─────────────────────────────────────────────────────┐")
print("  │  rdd_base                                           │")
print("  │  sc.parallelize(data, numSlices=4)                  │")
print("  │  12,000 registros · 4 particiones                   │")
print("  │  Tipo: ParallelCollectionRDD                        │")
print("  └──────────────────────┬──────────────────────────────┘")
print("                         │  NARROW (map)")
print("                         ▼")
print("  ┌─────────────────────────────────────────────────────┐")
print("  │  rdd_dag_meta                                       │")
print("  │  .map(lambda r: {image_id, label, label_name, ...}) │")
print("  │  12,000 registros · 4 particiones                   │")
print("  │  Tipo: MapPartitionsRDD                             │")
print("  └──────────────────────┬──────────────────────────────┘")
print("                         │  NARROW (filter)")
print("                         ▼")
print("  ┌─────────────────────────────────────────────────────┐")
print("  │  rdd_dag_bags                                       │")
print("  │  .filter(lambda r: r['label'] == 8)                 │")
print("  │  ≈ 1,200 registros · 4 particiones                  │")
print("  │  Tipo: MapPartitionsRDD                             │")
print("  └──────────────────────┬──────────────────────────────┘")
print("                         │  NARROW (map)")
print("                         ▼")
print("  ┌─────────────────────────────────────────────────────┐")
print("  │  rdd_dag_means                                      │")
print("  │  .map(lambda r: (image_id, pixel_mean))             │")
print("  │  ≈ 1,200 tuplas (image_id, mean) · 4 particiones    │")
print("  │  Tipo: MapPartitionsRDD                             │")
print("  └──────────────────────┬──────────────────────────────┘")
print("                         │  WIDE (sortBy → shuffle)")
print("                         ▼")
print("  ┌─────────────────────────────────────────────────────┐")
print("  │  rdd_dag_sorted                                     │")
print("  │  .sortBy(lambda x: x[1], ascending=True)            │")
print("  │  ≈ 1,200 tuplas ordenadas por pixel_mean            │")
print("  │  Tipo: ShuffledRDD (requiere shuffle entre nodos)   │")
print("  └──────────────────────┬──────────────────────────────┘")
print("                         │  ACCIÓN: .collect()")
print("                         ▼")
print("  ┌─────────────────────────────────────────────────────┐")
print("  │  Driver: lista Python con resultados                │")
print("  │  ← AQUÍ se dispara la ejecución del DAG completo   │")
print("  └─────────────────────────────────────────────────────┘")

print()
print("=" * 60)
print("PASO 4: Ejecutar .collect() y medir tiempo de ejecución")
print("=" * 60)

t0 = time.time()
resultados_dag = rdd_dag_sorted.collect()   # ACCIÓN: ejecuta el DAG completo
t1 = time.time()

tiempo_total = t1 - t0
print(f"\n   Ejecución completada en: {tiempo_total:.3f} segundos")
print(f"   Registros recuperados  : {len(resultados_dag):,}")
print(f"\n🔍 Top 5 imágenes de Bag más oscuras (menor pixel_mean):")
for image_id, mean_val in resultados_dag[:5]:
    print(f"   image_id={image_id:5d} | pixel_mean={mean_val:.6f}")
print(f"\n🔍 Top 5 imágenes de Bag más brillantes (mayor pixel_mean):")
for image_id, mean_val in resultados_dag[-5:]:
    print(f"   image_id={image_id:5d} | pixel_mean={mean_val:.6f}")

print(f"\n💡 Nota: En este ejemplo local, el 'costo' del pipeline es modesto.")
print(f"   En un cluster con millones de registros, el DAG optimizado")
print(f"   evita cómputos innecesarios aplicando filtros antes del sort.")

---
## Sección 8: `cache()` y `persist()`

### ¿Por qué cachear un RDD?

Por defecto, cada vez que se ejecuta una acción sobre un RDD, Spark **recalcula todo el linaje desde el principio**. Esto es costoso si el mismo RDD se usa en múltiples acciones.

```
Sin cache:
  Acción 1: rdd_base → map → filter → count()    ← recalcula
  Acción 2: rdd_base → map → filter → mean()     ← recalcula OTRA VEZ
  Acción 3: rdd_base → map → filter → stdev()    ← recalcula OTRA VEZ

Con cache:
  Acción 1: rdd_base → map → filter → count()    ← calcula y guarda en RAM
  Acción 2: (desde RAM) → mean()                 ← instantáneo
  Acción 3: (desde RAM) → stdev()                ← instantáneo
```

### `cache()` vs `persist()`:

```python
rdd.cache()                                   # ≡ persist(MEMORY_ONLY)
rdd.persist()                                 # ≡ persist(MEMORY_ONLY)
rdd.persist(StorageLevel.MEMORY_AND_DISK)     # RAM + disco si no cabe en RAM
rdd.persist(StorageLevel.DISK_ONLY)           # Solo disco (serializado)
rdd.persist(StorageLevel.MEMORY_ONLY_2)       # RAM con 2 réplicas (tolerancia a fallos)
```

### Niveles de almacenamiento:

| Nivel | RAM | Disco | Serializado | Réplicas | Cuándo usar |
|---|---|---|---|---|---|
| `MEMORY_ONLY` | ✅ | ❌ | ❌ | 1 | Dataset cabe en RAM |
| `MEMORY_AND_DISK` | ✅ | ✅ (overflow) | ❌ | 1 | Dataset puede no caber en RAM |
| `DISK_ONLY` | ❌ | ✅ | ✅ | 1 | RAM muy limitada |
| `MEMORY_ONLY_2` | ✅ | ❌ | ❌ | 2 | Alta disponibilidad |

> **Cuándo NO cachear**: si el RDD se usa solo una vez, no tiene sentido guardar en RAM.

In [ ]:
# ============================================================
# SECCIÓN 8: cache() y persist()
# ============================================================

print("=" * 60)
print("PASO 1: Cachear rdd_metadata con .cache()")
print("=" * 60)

# Importante: cache() es LAZY — solo marca el RDD para ser cacheado
# El caching real ocurre la PRIMERA VEZ que se ejecuta una acción
rdd_metadata_cached = rdd_metadata.cache()  # Marca para almacenar en RAM
print("   rdd_metadata.cache() → RDD marcado para ser cacheado en RAM (MEMORY_ONLY)")
print("   El cache aún no existe — se materializará en la primera acción")

print()
print("=" * 60)
print("PASO 2: Primera ejecución (frío) — cálculo + almacenamiento en cache")
print("=" * 60)

# Primera acción: Spark calcula el RDD Y lo almacena en cache
t_inicio_frio = time.time()
conteo_frio = rdd_metadata_cached.count()    # ACCIÓN: calcula + cachea
t_fin_frio   = time.time()
tiempo_frio  = t_fin_frio - t_inicio_frio

print(f"   count() = {conteo_frio:,}")
print(f"   ⏱️  Tiempo (frío, primera vez): {tiempo_frio:.4f} segundos")
print("   → Spark calculó las transformaciones Y guardó el resultado en RAM")

print()
print("=" * 60)
print("PASO 3: Segunda ejecución (caliente) — solo lectura desde cache")
print("=" * 60)

# Segunda acción: Spark lee desde cache en RAM (sin recalcular el linaje)
t_inicio_caliente = time.time()
conteo_caliente   = rdd_metadata_cached.count()   # ACCIÓN: lee desde cache
t_fin_caliente    = time.time()
tiempo_caliente   = t_fin_caliente - t_inicio_caliente

print(f"   count() = {conteo_caliente:,}")
print(f"   ⏱️  Tiempo (caliente, desde cache): {tiempo_caliente:.4f} segundos")
print("   → Spark leyó directamente desde RAM (sin recalcular el linaje)")

print()
print("=" * 60)
print("PASO 4: Comparación de tiempos")
print("=" * 60)

if tiempo_frio > 0:
    factor = tiempo_frio / tiempo_caliente if tiempo_caliente > 0 else float("inf")
else:
    factor = 1.0

print(f"\n   Tiempo sin cache (frío)    : {tiempo_frio:.4f} s")
print(f"   Tiempo con cache (caliente): {tiempo_caliente:.4f} s")
if factor > 1:
    print(f"   El cache fue {factor:.1f}x más rápido")
else:
    print(f"   Diferencia marginal (datos pequeños en modo local)")

print()
print("💡 Explicación de los resultados:")
print("   En modo LOCAL con datos pequeños, el cache puede no mostrar")
print("   una diferencia dramática porque:")
print("   a) No hay latencia de red entre nodos")
print("   b) Los datos ya están en la memoria del proceso Python")
print("   c) El overhead del scheduler de Spark es proporcional")
print()
print("   En un cluster REAL con datos en HDFS (terabytes):")
print("   ✅ Sin cache: leer disco + red + recalcular = minutos")
print("   ✅ Con cache: leer RAM = segundos (100-1000x más rápido)")

# También se puede usar persist() con nivel explícito
print()
print("   Ejemplo con persist() y nivel explícito:")
rdd_test_persist = rdd_test.persist(StorageLevel.MEMORY_AND_DISK)
_ = rdd_test_persist.count()  # Materializar
print(f"   rdd_test.persist(MEMORY_AND_DISK) → {rdd_test_persist.count():,} registros cacheados")

# Liberar cache para no consumir memoria innecesaria
rdd_metadata_cached.unpersist()
rdd_test_persist.unpersist()
print("\n   Cache liberado con .unpersist()")

---
## Sección 9: Cerrar Spark + Informe final

In [ ]:
# ============================================================
# SECCIÓN 9: CERRAR SPARK
# ============================================================

# Detener la SparkSession al finalizar
# Esto libera todos los recursos: memoria, threads, conexiones
spark.stop()
print("✅ SparkSession detenida correctamente")
print("   Todos los RDDs han sido eliminados de memoria")
print("   Los recursos del sistema han sido liberados")

---
## 📋 Resumen de la Lección 3

### Operaciones usadas en esta lección

| Operación | Tipo | ¿Lazy? | Código | Para qué sirve |
|---|---|---|---|---|
| `parallelize()` | Creación | ❌ | `sc.parallelize(data, 4)` | Crear RDD desde colección Python |
| `map()` | Transformación | ✅ | `rdd.map(lambda r: ...)` | Transformar elemento a elemento (1:1) |
| `filter()` | Transformación | ✅ | `rdd.filter(lambda r: cond)` | Seleccionar por condición booleana |
| `flatMap()` | Transformación | ✅ | `rdd.flatMap(lambda r: [...])` | Expandir y aplanar (1:N) |
| `distinct()` | Transformación | ✅ | `rdd.distinct()` | Eliminar duplicados |
| `sortBy()` | Transformación | ✅ | `rdd.sortBy(lambda x: x[1])` | Ordenar por clave o valor |
| `reduceByKey()` | Transformación | ✅ | `rdd.reduceByKey(lambda a,b: a+b)` | Agregar por clave (eficiente) |
| `mapValues()` | Transformación | ✅ | `rdd.mapValues(lambda v: v/n)` | Transformar solo los valores |
| `cache()` | Transformación | ✅ | `rdd.cache()` | Marcar para almacenar en RAM |
| `persist()` | Transformación | ✅ | `rdd.persist(StorageLevel.X)` | Almacenar con nivel explícito |
| `unpersist()` | Acción | ❌ | `rdd.unpersist()` | Liberar cache de RAM |
| `count()` | Acción | ❌ | `rdd.count()` | Contar todos los elementos |
| `first()` | Acción | ❌ | `rdd.first()` | Obtener el primer elemento |
| `take(n)` | Acción | ❌ | `rdd.take(5)` | Obtener los primeros N elementos |
| `collect()` | Acción | ❌ | `rdd.collect()` | Traer TODOS los datos al driver |
| `sum()` | Acción | ❌ | `rdd.sum()` | Sumar todos los valores numéricos |
| `mean()` | Acción | ❌ | `rdd.mean()` | Calcular la media |
| `stdev()` | Acción | ❌ | `rdd.stdev()` | Calcular la desviación estándar |
| `max()` / `min()` | Acción | ❌ | `rdd.max()` / `rdd.min()` | Valor máximo / mínimo |
| `toDebugString()` | Diagnóstico | ❌ | `rdd.toDebugString()` | Ver el linaje del RDD (DAG) |

---

### Conceptos clave aprendidos

1. **Lazy Evaluation**: Las transformaciones no se ejecutan hasta que se llama una acción. Esto permite a Spark optimizar el plan de ejecución completo.

2. **DAG (Directed Acyclic Graph)**: El linaje completo de transformaciones desde los datos originales hasta el resultado. Spark usa el DAG para optimizar y para recuperarse de fallos.

3. **Narrow vs Wide**: Las transformaciones _narrow_ (`map`, `filter`) no requieren shuffle. Las _wide_ (`sortBy`, `reduceByKey`, `distinct`) requieren mover datos entre particiones.

4. **Pair RDDs**: RDDs de tuplas `(clave, valor)` que habilitan operaciones de agrupación y agregación como `reduceByKey`.

5. **Cache/Persist**: Si un RDD se usa múltiples veces, almacenarlo en RAM evita recalcular el linaje repetidamente.

---

## 🔭 Conexión con Lección 4: DataFrames y Spark SQL

En la **Lección 4** daremos un salto de abstracción importante:

```python
# Lección 3 (RDD — bajo nivel):
rdd_class_means = rdd_base \
    .map(lambda r: (r['label_name'], (sum(r['pixels'])/784, 1))) \
    .reduceByKey(lambda a, b: (a[0]+b[0], a[1]+b[1])) \
    .mapValues(lambda v: v[0]/v[1])

# Lección 4 (DataFrame — alto nivel, mismo resultado):
df.groupBy("label_name") \
  .agg(F.mean("pixel_mean").alias("pixel_mean_avg"))
```

Los **DataFrames** añaden:
- **Esquema estructurado** (tipos de datos explícitos)
- **Catalyst Optimizer** (optimización automática de consultas)
- **Spark SQL** (consultas en SQL estándar)
- **Integración con pandas** (conversión directa con `.toPandas()`)

---

## ✅ Checklist de entregables — Lección 3

- [ ] **Sección 0**: SparkSession creada, Fashion-MNIST cargado, RDD base creado con 12,000 registros en 4 particiones
- [ ] **Sección 1**: Transformaciones `map` aplicadas: metadata, pixel_mean, normalización [-1,1]
- [ ] **Sección 2**: Transformaciones `filter` aplicadas: bags, bright, test
- [ ] **Sección 3**: Transformación `flatMap` aplicada: expansión de píxeles
- [ ] **Sección 4**: `distinct` y `sortBy` aplicados, bar chart generado
- [ ] **Sección 5**: Pair RDD creado, `reduceByKey` y `mapValues` aplicados
- [ ] **Sección 6**: Acciones ejecutadas: `count`, `sum`, `mean`, `stdev`, `max`, `min`
- [ ] **Sección 7**: DAG documentado con `toDebugString()` y representación ASCII
- [ ] **Sección 8**: `cache()` y `persist()` demostrados con medición de tiempos
- [ ] **Sección 9**: `spark.stop()` ejecutado correctamente

---
*RetailMax · Módulo 9 · Lección 3 · RDD Transformaciones y Acciones*